# Cat-Dog Image Classifier

### Importing Libraries

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory  # pyright: ignore
from tensorflow.keras import layers, Sequential # pyright: ignore
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint # pyright: ignore

I0000 00:00:1786811806.125657   13117 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786811806.925914   13117 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786811822.660548   13117 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


### Loading Images

In [ ]:
train = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'training',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

validation = image_dataset_from_directory(
    '../Data/catdog/training_set',
    validation_split = 0.2,
    subset = 'validation',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)


test = image_dataset_from_directory(
    '../Data/catdog/test_set',
    seed = 123,
    image_size = (180, 180),
    batch_size = 32
)

Found 8000 files belonging to 2 classes.
Using 6400 files for training.


W0000 00:00:1786811833.448083   13117 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Found 8000 files belonging to 2 classes.
Using 1600 files for validation.
Found 2000 files belonging to 2 classes.


### Scaling and augmenting 

In [ ]:
augmenting = Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])
rescale = layers.Rescaling(1./255)

### Building The Model

In [ ]:
model = Sequential([
    augmenting,
    rescale,
    layers.Conv2D(32,3, activation= 'relu', input_shape = (180,180,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(128,3, activation= 'relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation= 'relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation= 'sigmoid')
])

/mnt/c/Users/user/Work/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Compiling the model

In [ ]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

### Training the Model

In [ ]:
early_stop = EarlyStopping(monitor= 'val_loss', patience= 5, restore_best_weights= True)
checkpoint = ModelCheckpoint('best_model.keras', monitor= 'val_accuracy', save_best_only= True)

history = model.fit(
    train,
    validation_data  = validation,
    epochs = 20,
    callbacks = [early_stop, checkpoint]
)

Epoch 1/20


/mnt/c/Users/user/Work/.venv/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


200/200 ━━━━━━━━━━━━━━━━━━━━ 46s 224ms/step - accuracy: 0.5480 - loss: 0.7209 - val_accuracy: 0.5769 - val_loss: 0.6661
Epoch 2/20
200/200 ━━━━━━━━━━━━━━━━━━━━ 44s 221ms/step - accuracy: 0.5989 - loss: 0.6678 - val_accuracy: 0.6200 - val_loss: 0.6634
Epoch 3/20
119/200 ━━━━━━━━━━━━━━━━━━━━ 16s 202ms/step - accuracy: 0.6255 - loss: 0.6561